# JS06 (ANN) - Praktikum 6: Pengujian pada Fitur Musik Spotify
**Mata Kuliah:** Pembelajaran Mesin  
**Nama:** Aryaputra | **NIM:** 244107020127  

---
### Deskripsi
Pencarian tetangga terdekat pada fitur audio Spotify (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`) menggunakan StandardScaler dan perbandingan Exact NN, Annoy, dan HNSW.


In [1]:
import pandas as pd
import numpy as np
import time
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

features = ['danceability', 'energy', 'loudness', 'speechiness', 
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

np.random.seed(42)
n_songs = 3000
data_sp = {f: np.random.uniform(0, 1, n_songs) for f in features}
data_sp['loudness'] = np.random.uniform(-60, 0, n_songs)
data_sp['tempo'] = np.random.uniform(60, 200, n_songs)
df_sp = pd.DataFrame(data_sp)

X = df_sp[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

k = 10
query = X_scaled[:1]

# Exact NN
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean').fit(X_scaled)
start = time.time()
_, idx_exact = nn.kneighbors(query)
t_exact = time.time() - start

# Annoy
f = X_scaled.shape[1]
ann = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled): ann.add_item(i, v)
ann.build(10)
start = time.time()
idx_ann = ann.get_nns_by_vector(query[0], k)
t_ann = time.time() - start

# HNSW
p = hnswlib.Index(space='l2', dim=f)
p.init_index(max_elements=len(X_scaled), ef_construction=200, M=16)
p.add_items(X_scaled)
p.set_ef(100)
start = time.time()
idx_hnsw, _ = p.knn_query(query, k=k)
t_hnsw = time.time() - start

print(f"Exact NN 5 lagu terdekat : {idx_exact[0][:5]} ({t_exact*1000:.3f} ms)")
print(f"Annoy    5 lagu terdekat : {idx_ann[:5]} ({t_ann*1000:.3f} ms)")
print(f"HNSW     5 lagu terdekat : {idx_hnsw[0][:5]} ({t_hnsw*1000:.3f} ms)")


Exact NN 5 lagu terdekat : [   0  484 1949 2605 1071] (27.119 ms)
Annoy    5 lagu terdekat : [0, 484, 1949, 2605, 1071] (0.124 ms)
HNSW     5 lagu terdekat : [   0  484 1949 2605 1071] (0.790 ms)
